# 02 - Feature Engineering

**Goal:** Build a single customer-level feature matrix from all six raw datasets, ready for modeling.

Every feature group below is preceded by a **Why** block explaining the EDA finding that motivates it and how it connects to AML risk. The final cell produces `features.parquet` Ă˘â‚¬â€ť one row per customer.

> All features are computed from data that would be available *before* a compliance decision is made. No leakage from `suspicious_activity_confirmed` or post-hoc analyst decisions.

---
## 0. Setup & Data Load

In [1]:
import pandas as pd
import numpy as np

DATA = '../data/raw/'
OUT  = '../data/'

customers     = pd.read_csv(DATA + 'customers.csv')
transactions  = pd.read_csv(DATA + 'transactions.csv', parse_dates=['timestamp'])
accounts      = pd.read_csv(DATA + 'accounts.csv', parse_dates=['opening_date'])
baselines     = pd.read_csv(DATA + 'baselines.csv')
alert_history = pd.read_csv(DATA + 'alert_history.csv', parse_dates=['alert_date'])
country_risk  = pd.read_csv(DATA + 'country_risk.csv')

# Work only with approved transactions Ă˘â‚¬â€ť declined txns show attempted amounts, no funds moved
approved = transactions[transactions['status'] == 'approved'].copy()
approved['abs_amount'] = approved['amount'].abs()

print(f'Approved transactions: {len(approved):,}')
print(f'Customers: {len(customers):,}')

Approved transactions: 75,807
Customers: 1,200


---
## A. Customer Identity & Static Risk Signals

**Why:** EDA Section 3 showed that KYC rating, PEP status, and sanctions flags are already recorded per customer and require no computation. Despite being simple, they encode the bank's own risk assessment and regulatory obligations. PEP customers and those who ever matched on sanctions screening are subject to enhanced due diligence Ă˘â‚¬â€ť their presence in the suspicious group, even if small, is disproportionately important from a compliance standpoint.

Nationality mismatch (customer lives in DK but holds a foreign nationality) appeared as a modest signal in EDA and is standard practice in AML typologies Ă˘â‚¬â€ť it flags potential cross-border exposure that the bank's Danish-centric rules may underweight.

Customer type is encoded because corporate and SME customers have structurally different transaction patterns from personal customers Ă˘â‚¬â€ť mixing them without encoding would confuse the model.

In [2]:
feat_identity = customers[['customer_id', 'customer_type', 'kyc_risk_rating',
                             'pep_status', 'sanctions_screening_flag',
                             'nationality', 'residency_country',
                             'num_accounts', 'registration_date']].copy()

# Binary risk flags
feat_identity['pep_status']              = feat_identity['pep_status'].astype(int)
feat_identity['sanctions_flag']          = feat_identity['sanctions_screening_flag'].astype(int)
feat_identity['kyc_medium']              = (feat_identity['kyc_risk_rating'] == 'medium').astype(int)
feat_identity['nationality_mismatch']    = (
    feat_identity['nationality'] != feat_identity['residency_country']
).astype(int)

# Customer type dummies (personal is baseline)
type_dummies = pd.get_dummies(feat_identity['customer_type'], prefix='type', drop_first=False)
feat_identity = pd.concat([feat_identity, type_dummies], axis=1)

# Relationship age in days (longer relationship = more history; very new customers = less data)
feat_identity['registration_date'] = pd.to_datetime(feat_identity['registration_date'])
ref_date = pd.Timestamp('2025-01-01')  # day after the 12-month data window
feat_identity['relationship_days'] = (ref_date - feat_identity['registration_date']).dt.days

drop_cols = ['customer_type', 'kyc_risk_rating', 'sanctions_screening_flag',
             'nationality', 'residency_country', 'registration_date']
feat_identity.drop(columns=drop_cols, inplace=True)

print(feat_identity.shape)
feat_identity.head(3)

(1200, 11)


,customer_id,pep_status,num_accounts,sanctions_flag,kyc_medium,nationality_mismatch,type_SME,type_corporate,type_personal,type_sole_trader,relationship_days
0,CUST_0000,0,3,0,1,0,False,True,False,False,4895
1,CUST_0001,0,1,0,0,1,False,False,True,False,893
2,CUST_0002,0,3,0,0,0,False,False,False,True,1284


---
## B. Income vs. Observed Volume Consistency

**Why:** EDA Section 3 noted that declared annual income exists for personal customers and declared annual turnover for business customers, but both may be stale. Rather than trusting the declared figure as ground truth, we treat the *ratio* of observed transaction volume to declared income as a signal Ă˘â‚¬â€ť a customer moving 10x their declared income through accounts is a red flag regardless of whether the declaration is outdated.

This connects directly to one of the Case Brief's stated feature categories: 'income consistency'. It is one of the most interpretable AML signals Ă˘â‚¬â€ť easy to explain to a compliance analyst: 'this customer declared 400k DKK annual income but moved 4M DKK through their accounts this year'.

In [3]:
# Total annual volume from baselines (avg_monthly_volume * 12 is a reasonable proxy)
vol = baselines[['customer_id', 'avg_monthly_volume']].copy()
vol['annual_volume_proxy'] = vol['avg_monthly_volume'] * 12

# Declared income/turnover Ă˘â‚¬â€ť use whichever is available
inc = customers[['customer_id', 'customer_type',
                  'declared_annual_income', 'declared_annual_turnover']].copy()
inc['declared_annual'] = np.where(
    inc['customer_type'].isin(['corporate', 'SME', 'sole_trader']),
    inc['declared_annual_turnover'],
    inc['declared_annual_income']
)

feat_income = inc[['customer_id', 'declared_annual']].merge(vol, on='customer_id', how='left')

# Ratio: observed volume / declared income (clip to avoid inf; nulls become NaN -> impute later)
feat_income['income_volume_ratio'] = (
    feat_income['annual_volume_proxy'] / (feat_income['declared_annual'] + 1)
).clip(upper=1000)

# Flag: declared income is null (common for corporates Ă˘â‚¬â€ť handled separately by model)
feat_income['income_declared_missing'] = feat_income['declared_annual'].isna().astype(int)

feat_income = feat_income[['customer_id', 'income_volume_ratio', 'income_declared_missing']]

print(feat_income['income_volume_ratio'].describe().round(2))
feat_income.head(3)

count    1142.00
mean        1.86
std         4.99
min         0.01
25%         1.04
50%         1.26
75%         1.49
max        83.43
Name: income_volume_ratio, dtype: float64


,customer_id,income_volume_ratio,income_declared_missing
0,CUST_0000,1.000890,0
1,CUST_0001,83.427802,0
2,CUST_0002,NaN,1


---
## C. Transaction Velocity & Volume

**Why:** EDA Section 4 showed that suspicious customers have visibly different distributions of transaction count, total volume, and max single transaction Ă˘â‚¬â€ť even at the customer level. High transaction velocity combined with high volume is a core AML red flag: it suggests money is moving in and out rapidly rather than sitting in accounts as you'd expect from legitimate savings or business activity.

We compute these directly from raw transactions rather than relying solely on `baselines.csv` because the baseline aggregates cover only 6 months Ă˘â‚¬â€ť the transactions table covers the full 12-month window, giving us more signal and the ability to detect trends across the year.

Declined transaction count is included separately: a high ratio of declined transactions can indicate probing behaviour Ă˘â‚¬â€ť testing which amounts or channels get through.

In [4]:
txn_agg = approved.groupby('customer_id').agg(
    txn_count_total=('transaction_id', 'count'),
    total_volume=('abs_amount', 'sum'),
    avg_txn_amount=('abs_amount', 'mean'),
    max_single_txn=('abs_amount', 'max'),
    std_txn_amount=('abs_amount', 'std'),
).reset_index()

# Declined transaction count
declined = transactions[transactions['status'] == 'declined']
declined_counts = declined.groupby('customer_id').size().reset_index(name='declined_txn_count')
txn_agg = txn_agg.merge(declined_counts, on='customer_id', how='left')
txn_agg['declined_txn_count'] = txn_agg['declined_txn_count'].fillna(0)

# Declined ratio: declined / (approved + declined)
txn_agg['declined_ratio'] = (
    txn_agg['declined_txn_count'] /
    (txn_agg['txn_count_total'] + txn_agg['declined_txn_count'])
)

# Coefficient of variation: std / mean Ă˘â‚¬â€ť high CV means erratic spending, not routine
txn_agg['txn_amount_cv'] = txn_agg['std_txn_amount'] / (txn_agg['avg_txn_amount'] + 1)

feat_velocity = txn_agg[['customer_id', 'txn_count_total', 'total_volume',
                           'avg_txn_amount', 'max_single_txn', 'txn_amount_cv',
                           'declined_txn_count', 'declined_ratio']]

print(feat_velocity.shape)
feat_velocity.head(3)

(1200, 8)


,customer_id,txn_count_total,total_volume,avg_txn_amount,max_single_txn,txn_amount_cv,declined_txn_count,declined_ratio
0,CUST_0000,84,48687454.88,579612.558095,5825558.08,1.856269,0.0,0.000000
1,CUST_0001,154,7102237.52,46118.425455,120065.45,0.756206,1.0,0.006452
2,CUST_0002,62,677698.66,10930.623548,47289.43,1.597565,3.0,0.046154


---
## D. Structuring Signal

**Why:** EDA Section 4 explicitly confirmed this Ă˘â‚¬â€ť cash transaction amounts cluster just below NordikBank's internal 15,000 DKK reporting threshold. This is the textbook definition of structuring (also called 'smurfing'): deliberately keeping individual transactions below a threshold to avoid detection.

The Case Brief calls this out by name: 'structuring patterns' is one of the six trigger rules in the existing TMS. However, the current system applies this as a static rule on individual transactions. We improve on it by computing it as a *customer-level behavioural pattern* Ă˘â‚¬â€ť how many of a customer's cash transactions fall in the suspicious band, and what fraction of their total cash activity that represents.

The band [13,000Ă˘â‚¬â€ś14,999 DKK] is chosen to capture amounts meaningfully below the threshold while excluding genuinely round amounts that may be legitimate (e.g. a 14,000 DKK monthly rent).

In [5]:
cash = approved[approved['transaction_type'].isin(['cash_deposit', 'cash_withdrawal'])].copy()

# Structuring band: just below the 15,000 DKK internal threshold
cash['in_structuring_band'] = (
    (cash['abs_amount'] >= 13000) & (cash['abs_amount'] < 15000)
).astype(int)

struct_agg = cash.groupby('customer_id').agg(
    cash_txn_count=('transaction_id', 'count'),
    structuring_count=('in_structuring_band', 'sum'),
    cash_total_volume=('abs_amount', 'sum'),
).reset_index()

struct_agg['structuring_ratio'] = (
    struct_agg['structuring_count'] / struct_agg['cash_txn_count']
)

# Customers with no cash transactions get zeros
all_customers = customers[['customer_id']].copy()
feat_structuring = all_customers.merge(struct_agg, on='customer_id', how='left').fillna(0)

print(f"Customers with at least 1 structuring-band cash txn: {(feat_structuring['structuring_count'] > 0).sum()}")
feat_structuring.head(3)

Customers with at least 1 structuring-band cash txn: 55


,customer_id,cash_txn_count,structuring_count,cash_total_volume,structuring_ratio
0,CUST_0000,24.0,13.0,302300.0,0.541667
1,CUST_0001,4.0,0.0,7200.0,0.000000
2,CUST_0002,0.0,0.0,0.0,0.000000


---
## E. Cash Intensity

**Why:** Cash is the hardest transaction type to trace Ă˘â‚¬â€ť it leaves no counterparty record and no digital trail. EDA Section 4 showed that the share of cash transactions differs between suspicious and clean customers. The existing TMS flags individual large cash transactions, but misses customers who accumulate many smaller ones.

We compute cash intensity as the fraction of a customer's total transaction count and total volume that is cash-based. A high cash share relative to a customer's declared occupation (e.g. a software engineer with 40% cash transactions) is anomalous. We also split by direction Ă˘â‚¬â€ť cash deposits are more suspicious than cash withdrawals in most layering typologies, as they represent unexplained funds entering the system.

In [6]:
cash_dir = approved[approved['transaction_type'].isin(['cash_deposit', 'cash_withdrawal'])].copy()

cash_by_dir = cash_dir.groupby(['customer_id', 'transaction_type']).agg(
    count=('transaction_id', 'count'),
    volume=('abs_amount', 'sum'),
).unstack(fill_value=0)
cash_by_dir.columns = ['_'.join(c) for c in cash_by_dir.columns]
cash_by_dir = cash_by_dir.reset_index()

# Merge with total txn counts to compute ratios
cash_feats = customers[['customer_id']].merge(
    cash_by_dir, on='customer_id', how='left'
).merge(
    feat_velocity[['customer_id', 'txn_count_total', 'total_volume']], on='customer_id', how='left'
).fillna(0)

# Rename direction columns robustly
dir_cols = {c: c for c in cash_feats.columns}
cash_feats.rename(columns={
    c: c for c in cash_feats.columns
}, inplace=True)

for col in ['count_cash_deposit', 'count_cash_withdrawal', 'volume_cash_deposit', 'volume_cash_withdrawal']:
    if col not in cash_feats.columns:
        cash_feats[col] = 0

cash_feats['cash_txn_count'] = cash_feats['count_cash_deposit'] + cash_feats['count_cash_withdrawal']
cash_feats['cash_volume']    = cash_feats['volume_cash_deposit'] + cash_feats['volume_cash_withdrawal']

cash_feats['pct_cash_txns']   = cash_feats['cash_txn_count']  / (cash_feats['txn_count_total'] + 1)
cash_feats['pct_cash_volume'] = cash_feats['cash_volume']     / (cash_feats['total_volume'] + 1)
cash_feats['pct_cash_deposits_of_cash'] = (
    cash_feats['count_cash_deposit'] / (cash_feats['cash_txn_count'] + 1)
)

feat_cash = cash_feats[['customer_id', 'pct_cash_txns', 'pct_cash_volume', 'pct_cash_deposits_of_cash']]

print(feat_cash.describe().round(3))
feat_cash.head(3)

       pct_cash_txns  pct_cash_volume  pct_cash_deposits_of_cash
count       1200.000         1200.000                   1200.000
mean           0.075            0.029                      0.102
std            0.119            0.107                      0.235
min            0.000            0.000                      0.000
25%            0.014            0.000                      0.000
50%            0.043            0.005                      0.000
75%            0.088            0.015                      0.000
max            0.845            0.981                      0.992


,customer_id,pct_cash_txns,pct_cash_volume,pct_cash_deposits_of_cash
0,CUST_0000,0.282353,0.006209,0.96
1,CUST_0001,0.025806,0.001014,0.00
2,CUST_0002,0.000000,0.000000,0.00


---
## F. Counterparty Network Diversity

**Why:** EDA Section 5 (baselines) showed `num_unique_counterparties_6m` has meaningful discriminative power. The Case Brief explicitly lists 'counterparty networks' as a feature engineering target.

A legitimate customer tends to transact with a stable set of counterparties (employer, landlord, utilities, a few shops). A suspicious customer may show two opposite patterns: extremely many unique counterparties (dispersing funds widely) or extremely few (funnelling to a single recipient). Both patterns are suspicious for different reasons Ă˘â‚¬â€ť high diversity can indicate placement, while concentration can indicate layering to a shell company.

We also flag whether a customer has any recurring counterparty relationship vs. entirely one-off transactions, since recurring payments (standing orders, direct debits) are generally consistent with legitimate economic activity.

In [7]:
cp_agg = approved.groupby('customer_id').agg(
    unique_counterparties=('counterparty_id', 'nunique'),
    recurring_txn_count=('is_recurring', 'sum'),
    total_txns=('transaction_id', 'count'),
).reset_index()

cp_agg['pct_recurring'] = cp_agg['recurring_txn_count'] / (cp_agg['total_txns'] + 1)

# Counterparty concentration: max share any single counterparty represents
cp_share = (
    approved.groupby(['customer_id', 'counterparty_id'])
    .size()
    .reset_index(name='cp_count')
)
cp_total = approved.groupby('customer_id').size().reset_index(name='total')
cp_share = cp_share.merge(cp_total, on='customer_id')
cp_share['cp_share'] = cp_share['cp_count'] / cp_share['total']
cp_max_share = cp_share.groupby('customer_id')['cp_share'].max().reset_index(name='max_counterparty_share')

feat_counterparty = customers[['customer_id']].merge(
    cp_agg[['customer_id', 'unique_counterparties', 'pct_recurring']], on='customer_id', how='left'
).merge(cp_max_share, on='customer_id', how='left').fillna(0)

print(feat_counterparty.describe().round(3))
feat_counterparty.head(3)

       unique_counterparties  pct_recurring  max_counterparty_share
count               1200.000       1200.000                1200.000
mean                  10.765          0.560                   0.221
std                    5.868          0.168                   0.078
min                    3.000          0.024                   0.058
25%                    7.000          0.466                   0.174
50%                    9.000          0.609                   0.204
75%                   12.000          0.679                   0.255
max                   35.000          0.932                   0.587


,customer_id,unique_counterparties,pct_recurring,max_counterparty_share
0,CUST_0000,23,0.211765,0.142857
1,CUST_0001,12,0.167742,0.357143
2,CUST_0002,12,0.682540,0.161290


---
## G. Geographic & Country Risk Exposure

**Why:** EDA Section 8 confirmed that suspicious customers have higher exposure to international counterparties and lower average CPI scores on those counterparties. The Data Dictionary explicitly warns that `counterparty_bank_country` is null for ~97% of transactions Ă˘â‚¬â€ť only international wires are populated. This means the signal is sparse but strong when present: a customer who sends international wires to high-risk jurisdictions is flagging clearly.

We encode this at three levels:
1. **Volume level** Ă˘â‚¬â€ť what share of transactions are international wires at all
2. **Risk level** Ă˘â‚¬â€ť of those, what share go to EU high-risk listed countries
3. **Customer identity level** Ă˘â‚¬â€ť does the customer's own nationality carry elevated risk

The minimum CPI across all counterparty countries captures worst-case exposure, which matters more than average CPI in a risk context: one connection to a high-corruption jurisdiction is a red flag regardless of other clean connections.

In [8]:
# International wire transactions only
intl = approved[approved['counterparty_bank_country'].notna()].copy()
intl = intl.merge(country_risk, left_on='counterparty_bank_country', right_on='country_code', how='left')

intl_agg = intl.groupby('customer_id').agg(
    intl_txn_count=('transaction_id', 'count'),
    intl_volume=('abs_amount', 'sum'),
    high_risk_country_txns=('eu_high_risk_list', 'sum'),
    min_cpi=('corruption_perception_index', 'min'),
    unique_countries=('counterparty_bank_country', 'nunique'),
).reset_index()

intl_agg['pct_intl_high_risk'] = (
    intl_agg['high_risk_country_txns'] / (intl_agg['intl_txn_count'] + 1)
)

# Merge with total counts for share features
feat_geo = customers[['customer_id', 'nationality']].merge(
    feat_velocity[['customer_id', 'txn_count_total']], on='customer_id', how='left'
).merge(intl_agg, on='customer_id', how='left')

feat_geo['intl_txn_count']    = feat_geo['intl_txn_count'].fillna(0)
feat_geo['intl_volume']        = feat_geo['intl_volume'].fillna(0)
feat_geo['unique_countries']   = feat_geo['unique_countries'].fillna(0)
feat_geo['pct_intl_high_risk'] = feat_geo['pct_intl_high_risk'].fillna(0)
feat_geo['high_risk_country_txns'] = feat_geo['high_risk_country_txns'].fillna(0)

# Customer's nationality risk
feat_geo = feat_geo.merge(
    country_risk[['country_code', 'eu_high_risk_list', 'corruption_perception_index']]
        .rename(columns={'country_code': 'nationality',
                         'eu_high_risk_list': 'nationality_high_risk',
                         'corruption_perception_index': 'nationality_cpi'}),
    on='nationality', how='left'
)
feat_geo['nationality_high_risk'] = feat_geo['nationality_high_risk'].fillna(0).astype(int)
feat_geo['nationality_cpi']       = feat_geo['nationality_cpi'].fillna(feat_geo['nationality_cpi'].median())

# pct of all txns that are international
feat_geo['pct_intl_txns'] = feat_geo['intl_txn_count'] / (feat_geo['txn_count_total'] + 1)

feat_geo = feat_geo[['customer_id', 'pct_intl_txns', 'unique_countries',
                      'pct_intl_high_risk', 'high_risk_country_txns',
                      'min_cpi', 'nationality_high_risk', 'nationality_cpi']]

print(feat_geo.describe().round(3))
feat_geo.head(3)

       pct_intl_txns  unique_countries  pct_intl_high_risk  \
count       1200.000          1200.000            1200.000   
mean           0.029             1.062               0.054   
std            0.065             1.783               0.164   
min            0.000             0.000               0.000   
25%            0.000             0.000               0.000   
50%            0.000             0.000               0.000   
75%            0.024             1.000               0.000   
max            0.600            10.000               0.952   

       high_risk_country_txns  min_cpi  nationality_high_risk  nationality_cpi  
count                1200.000  522.000               1200.000         1200.000  
mean                    0.456   45.579                  0.009           81.942  
std                     2.752   16.928                  0.095           13.892  
min                     0.000   24.000                  0.000           27.000  
25%                     0.000   33.0

,customer_id,pct_intl_txns,unique_countries,pct_intl_high_risk,high_risk_country_txns,min_cpi,nationality_high_risk,nationality_cpi
0,CUST_0000,0.058824,4.0,0.0,0.0,67.0,0,88
1,CUST_0001,0.000000,0.0,0.0,0.0,NaN,0,71
2,CUST_0002,0.000000,0.0,0.0,0.0,NaN,0,88


---
## H. Account Structure & Dormancy

**Why:** EDA Section 7 showed that dormant accounts (zero balance, zero flow) are a meaningful signal Ă˘â‚¬â€ť they represent accounts that exist but are not being used in a normal way. In AML typologies, dormant accounts that suddenly activate are one of the six trigger rules in NordikBank's existing TMS (`dormant_activation`). We improve on the rule by quantifying the degree of dormancy across all of a customer's accounts.

The inflow/outflow ratio captures layering behaviour: money enters the account system but does not leave through normal economic channels (bills, purchases, transfers to self). A high ratio suggests funds are accumulating in ways inconsistent with stated income.

Having a foreign-currency account is also encoded Ă˘â‚¬â€ť it is legitimate for international businesses, but anomalous for a personal customer with no declared international activity.

In [9]:
accounts['is_dormant'] = (
    (accounts['avg_monthly_balance_6m'] == 0) &
    (accounts['avg_monthly_inflow_6m'] == 0) &
    (accounts['avg_monthly_outflow_6m'] == 0)
).astype(int)

acc_agg = accounts.groupby('customer_id').agg(
    num_accounts=('account_id', 'count'),
    num_dormant=('is_dormant', 'sum'),
    has_foreign_currency=('currency', lambda x: int((x != 'DKK').any())),
    total_balance=('avg_monthly_balance_6m', 'sum'),
    total_inflow=('avg_monthly_inflow_6m', 'sum'),
    total_outflow=('avg_monthly_outflow_6m', 'sum'),
).reset_index()

acc_agg['pct_dormant_accounts']  = acc_agg['num_dormant'] / acc_agg['num_accounts']
acc_agg['inflow_outflow_ratio']  = acc_agg['total_inflow'] / (acc_agg['total_outflow'] + 1)
acc_agg['balance_to_inflow']     = acc_agg['total_balance'] / (acc_agg['total_inflow'] + 1)

feat_accounts = acc_agg[['customer_id', 'num_accounts', 'num_dormant',
                           'pct_dormant_accounts', 'has_foreign_currency',
                           'inflow_outflow_ratio', 'balance_to_inflow']]

print(feat_accounts.describe().round(3))
feat_accounts.head(3)

       num_accounts  num_dormant  pct_dormant_accounts  has_foreign_currency  \
count      1200.000     1200.000              1200.000              1200.000   
mean          2.252        1.252                 0.438                 0.229   
std           1.064        1.064                 0.280                 0.420   
min           1.000        0.000                 0.000                 0.000   
25%           1.000        0.000                 0.000                 0.000   
50%           2.000        1.000                 0.500                 0.000   
75%           3.000        2.000                 0.667                 0.000   
max           5.000        4.000                 0.800                 1.000   

       inflow_outflow_ratio  balance_to_inflow  
count              1200.000           1200.000  
mean                121.964              1.753  
std                 502.633              0.496  
min                   0.229              0.300  
25%                   2.431       

,customer_id,num_accounts,num_dormant,pct_dormant_accounts,has_foreign_currency,inflow_outflow_ratio,balance_to_inflow
0,CUST_0000,3,2,0.666667,0,3113.205456,2.299358
1,CUST_0001,1,0,0.000000,0,1.285773,0.744519
2,CUST_0002,3,2,0.666667,0,8.590103,2.067173


---
## I. Temporal Behaviour Patterns

**Why:** EDA Section 4 showed hour-of-day distributions differ between suspicious and clean customers. The baselines already provide `transaction_time_entropy` Ă˘â‚¬â€ť we verify its signal and supplement it with features the baseline cannot capture over the full 12-month window.

Burst activity Ă˘â‚¬â€ť many transactions in a short period followed by silence Ă˘â‚¬â€ť is a known AML pattern. It appears in the TMS as the `rapid_movement` trigger rule. We operationalise it as the maximum number of transactions in any 7-day rolling window, normalised by the customer's typical monthly count.

Month-to-month volume variance captures erratic activity: legitimate customers tend to have stable monthly flows (salary, rent, regular bills), while suspicious customers may show sudden spikes. We use coefficient of variation across monthly volumes as the measure.

In [10]:
approved['month'] = approved['timestamp'].dt.to_period('M')
approved['date']  = approved['timestamp'].dt.date

# Monthly volume CV: how erratic are monthly totals?
monthly_vol = approved.groupby(['customer_id', 'month'])['abs_amount'].sum().reset_index()
monthly_stats = monthly_vol.groupby('customer_id')['abs_amount'].agg(['mean', 'std']).reset_index()
monthly_stats.columns = ['customer_id', 'monthly_vol_mean', 'monthly_vol_std']
monthly_stats['monthly_vol_cv'] = monthly_stats['monthly_vol_std'] / (monthly_stats['monthly_vol_mean'] + 1)

# Monthly count CV
monthly_cnt = approved.groupby(['customer_id', 'month']).size().reset_index(name='cnt')
monthly_cnt_stats = monthly_cnt.groupby('customer_id')['cnt'].agg(['mean', 'std']).reset_index()
monthly_cnt_stats.columns = ['customer_id', 'monthly_cnt_mean', 'monthly_cnt_std']
monthly_cnt_stats['monthly_cnt_cv'] = (
    monthly_cnt_stats['monthly_cnt_std'] / (monthly_cnt_stats['monthly_cnt_mean'] + 1)
)

# Peak 7-day transaction count (burst detection)
daily_cnt = approved.groupby(['customer_id', 'date']).size().reset_index(name='daily_cnt')
daily_cnt['date'] = pd.to_datetime(daily_cnt['date'])

def rolling_7d_max(grp):
    grp = grp.set_index('date').sort_index()
    return grp['daily_cnt'].rolling('7D').sum().max()

peak_7d = daily_cnt.groupby('customer_id').apply(rolling_7d_max).reset_index(name='peak_7d_txn_count')

# Off-hours activity: share of txns between 23:00 and 05:00
approved['hour'] = approved['timestamp'].dt.hour
approved['is_offhours'] = approved['hour'].between(23, 5) | (approved['hour'] < 5)
offhours = approved.groupby('customer_id').agg(
    offhours_count=('is_offhours', 'sum'),
    total=('transaction_id', 'count'),
).reset_index()
offhours['pct_offhours'] = offhours['offhours_count'] / offhours['total']

feat_temporal = customers[['customer_id']].merge(
    monthly_stats[['customer_id', 'monthly_vol_cv']], on='customer_id', how='left'
).merge(
    monthly_cnt_stats[['customer_id', 'monthly_cnt_cv']], on='customer_id', how='left'
).merge(
    peak_7d, on='customer_id', how='left'
).merge(
    offhours[['customer_id', 'pct_offhours']], on='customer_id', how='left'
).fillna(0)

print(feat_temporal.describe().round(3))
feat_temporal.head(3)

       monthly_vol_cv  monthly_cnt_cv  peak_7d_txn_count  pct_offhours
count        1200.000        1200.000           1200.000      1200.000
mean            0.212           0.271              4.913         0.385
std             0.306           0.087              1.394         0.139
min             0.012           0.084              2.000         0.000
25%             0.037           0.213              4.000         0.300
50%             0.149           0.259              5.000         0.406
75%             0.310           0.315              5.000         0.489
max             3.349           0.867             12.000         0.674


C:\Users\kubad\AppData\Local\Temp\ipykernel_36180\2530293099.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  peak_7d = daily_cnt.groupby('customer_id').apply(rolling_7d_max).reset_index(name='peak_7d_txn_count')


,customer_id,monthly_vol_cv,monthly_cnt_cv,peak_7d_txn_count,pct_offhours
0,CUST_0000,0.252320,0.266501,5.0,0.214286
1,CUST_0001,0.349899,0.215402,10.0,0.090909
2,CUST_0002,0.458611,0.271841,3.0,0.532258


---
## J. Baseline Features (Pass-Through)

**Why:** EDA Section 5 showed that the pre-computed baseline features in `baselines.csv` already have meaningful correlation with the label Ă˘â‚¬â€ť particularly `avg_monthly_volume`, `pct_cash_transactions`, `geographic_spread_score`, and `dormancy_periods_count`. Rather than re-deriving all of them from scratch, we pass them through directly as a foundation and let the model decide how to weight them alongside our engineered features.

This is also a useful sanity check: if our engineered features from the raw transactions don't outperform or complement these pre-computed ones, we need to revisit our feature engineering approach. The baselines covering 6 months while our features cover 12 months means the two sets should carry complementary temporal signal.

In [11]:
feat_baselines = baselines.copy()

print(feat_baselines.shape)
feat_baselines.head(3)

(1200, 10)


,customer_id,avg_monthly_transaction_count,avg_monthly_volume,max_single_transaction_6m,pct_international_transactions,pct_cash_transactions,num_unique_counterparties_6m,transaction_time_entropy,geographic_spread_score,dormancy_periods_count
0,CUST_0000,6.8,3.483933e+06,5.794449e+06,0.0833,0.3185,11.8124,2.9906,2.2102,0
1,CUST_0001,13.8,7.786664e+05,1.095073e+05,0.0000,0.0240,12.1176,1.9788,0.0000,0
2,CUST_0002,5.0,6.125753e+04,4.271092e+04,0.0000,0.0000,7.5451,2.2957,0.0000,1


---
## K. Deviation from Baseline (12-Month vs 6-Month)

**Why:** SKILLS.md explicitly notes that raw transaction values are nearly meaningless without baseline context Ă˘â‚¬â€ť a 500,000 DKK transaction means something different for a customer whose baseline max is 5M DKK vs. one whose max is 50,000 DKK. This is the core insight behind behavioural deviation features.

By computing how much the full 12-month observed behaviour deviates from the 6-month baseline, we capture *change* Ă˘â‚¬â€ť a customer who was quiet for 6 months and then became highly active in the second half of the year will show a large positive deviation. This is exactly the dormant-activation pattern the TMS tries to catch, but operationalised as a continuous score rather than a binary rule.

In [12]:
# Volume deviation: 12-month total vs 6-month baseline annualised
dev = feat_velocity[['customer_id', 'total_volume', 'txn_count_total']].merge(
    baselines[['customer_id', 'avg_monthly_volume', 'avg_monthly_transaction_count',
               'max_single_transaction_6m']], on='customer_id', how='left'
)

dev['baseline_annual_vol']   = dev['avg_monthly_volume'] * 12
dev['baseline_annual_count'] = dev['avg_monthly_transaction_count'] * 12

# Ratio: actual / baseline (>1 means more active than baseline, <1 means quieter)
dev['vol_vs_baseline']   = dev['total_volume']       / (dev['baseline_annual_vol'] + 1)
dev['count_vs_baseline'] = dev['txn_count_total']    / (dev['baseline_annual_count'] + 1)
dev['max_vs_baseline']   = dev['total_volume']       / (dev['max_single_transaction_6m'] + 1)

feat_deviation = dev[['customer_id', 'vol_vs_baseline', 'count_vs_baseline', 'max_vs_baseline']]

print(feat_deviation.describe().round(3))
feat_deviation.head(3)

       vol_vs_baseline  count_vs_baseline  max_vs_baseline
count         1200.000           1200.000         1200.000
mean             1.049              0.959           16.643
std              0.972              0.121           16.234
min              0.328              0.433            1.728
25%              0.898              0.895           11.857
50%              0.987              0.961           14.471
75%              1.092              1.029           17.400
max             28.624              1.589          392.416


,customer_id,vol_vs_baseline,count_vs_baseline,max_vs_baseline
0,CUST_0000,1.164571,1.016949,8.402430
1,CUST_0001,0.760085,0.924370,64.855719
2,CUST_0002,0.921924,1.016393,15.866734


---
## L. Transaction Type Mix

**Why:** EDA Section 4 showed the transaction type mix visibly differs between suspicious and clean customers. 
Cash intensity (Group E) captures only cash-related types. 
But the full mix matters: international wires are the highest-risk type (only 3.1% of all transactions but directly linked to cross-border exposure), 
while transfers are the most common type and show a different suspicious vs clean split than direct debits or card payments.

Each feature here is the share of a customer's approved transactions that fall into a given type. 
Shares sum to 1 across types, so the model can detect when a customer's mix is anomalous relative to their segment. 
A corporate customer with 0% card payments and 40% international wires looks very different from one with the typical mix. 
These are cheap to compute and directly motivated by the EDA type mix bar chart.

In [ ]:
type_counts = approved.groupby(['customer_id', 'transaction_type']).size().unstack(fill_value=0)
type_counts.columns = [f'txn_type_{c}' for c in type_counts.columns]
type_counts = type_counts.reset_index()

# Normalise to shares
type_cols = [c for c in type_counts.columns if c.startswith('txn_type_')]
row_totals = type_counts[type_cols].sum(axis=1)
type_counts[type_cols] = type_counts[type_cols].div(row_totals, axis=0)

feat_type_mix = customers[['customer_id']].merge(type_counts, on='customer_id', how='left').fillna(0)

print('Type mix share features:')
print(feat_type_mix[type_cols].describe().round(3))
feat_type_mix.head(3)

---
## L. Assemble Final Feature Matrix

**Why:** All feature groups are joined on `customer_id` using left joins from the customer master list. This guarantees one row per customer Ă˘â‚¬â€ť including test-set customers who have no label yet. Missing values after joining (e.g. a customer with no international transactions having a null `min_cpi`) are imputed with sensible defaults: 0 for count/share features, median for continuous risk scores.

The label and split columns are attached for convenience but are **not** features Ă˘â‚¬â€ť they must be excluded before passing to any model.

In [17]:
feature_groups = [
    feat_identity,
    feat_income,
    feat_velocity,
    feat_structuring,
    feat_cash,
    feat_counterparty,
    feat_geo,
    feat_accounts,
    feat_temporal,
    feat_baselines,
    feat_deviation,
    feat_type_mix,
]

features = customers[["customer_id", "suspicious_activity_confirmed", "split"]].copy()

for grp in feature_groups:
    features = features.merge(grp, on="customer_id", how="left")

# num_accounts appears in both feat_identity and feat_accounts — same value, drop the duplicate
if "num_accounts_x" in features.columns and "num_accounts_y" in features.columns:
    features.drop(columns=["num_accounts_y"], inplace=True)
    features.rename(columns={"num_accounts_x": "num_accounts"}, inplace=True)

# Drop any remaining duplicate columns
features = features.loc[:, ~features.columns.duplicated()]

print("Feature matrix shape:", features.shape)
print("Null counts per column:")
nulls = features.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False))

NameError: name 'feat_type_mix' is not defined

---
## M. Imputation

**Why:** Remaining nulls fall into two categories:

1. **Structural nulls** Ă˘â‚¬â€ť e.g. `min_cpi` is null for customers with no international transactions. These should be imputed with a neutral/safe value (median CPI for missing, 0 for counts).
2. **Missing declared income** Ă˘â‚¬â€ť already flagged as `income_declared_missing`. Impute the ratio with 0 (no ratio can be computed) and let the flag column carry the signal.

We do **not** impute `suspicious_activity_confirmed` Ă˘â‚¬â€ť nulls there are the test set and are correct.

In [14]:
# Columns to impute with 0 (counts, shares, flags)
zero_impute = [
    'intl_txn_count', 'intl_volume', 'unique_countries', 'pct_intl_high_risk',
    'high_risk_country_txns', 'pct_intl_txns', 'structuring_count', 'structuring_ratio',
    'cash_txn_count', 'cash_total_volume',
]
for col in zero_impute:
    if col in features.columns:
        features[col] = features[col].fillna(0)

# min_cpi: impute with median (neutral Ă˘â‚¬â€ť unknown risk is not zero risk)
if 'min_cpi' in features.columns:
    features['min_cpi'] = features['min_cpi'].fillna(features['min_cpi'].median())

# income_volume_ratio: impute with 0 when income is unknown
if 'income_volume_ratio' in features.columns:
    features['income_volume_ratio'] = features['income_volume_ratio'].fillna(0)

# All remaining numeric nulls: median imputation
num_cols = features.select_dtypes(include='number').columns.tolist()
num_cols = [c for c in num_cols if c != 'suspicious_activity_confirmed']
for col in num_cols:
    if features[col].isnull().any():
        features[col] = features[col].fillna(features[col].median())

print(f'Remaining nulls after imputation:')
remaining = features.isnull().sum()
print(remaining[remaining > 0] if remaining.any() else 'None')

Remaining nulls after imputation:
suspicious_activity_confirmed    500
dtype: int64


---
## N. Save & Summary

**Why:** Saving as parquet preserves dtypes (especially booleans and categoricals) and is significantly faster to read back than CSV for a 1,200-row wide matrix. The modeling notebook loads this file directly.

In [15]:
import os
os.makedirs(OUT, exist_ok=True)

# Exclude object columns (customer_id stays as index reference, not a feature)
features.to_parquet(OUT + 'features.parquet', index=False)

feature_cols = [c for c in features.columns
                if c not in ['customer_id', 'suspicious_activity_confirmed', 'split']
                and features[c].dtype != object]

print(f'Saved features.parquet')
print(f'Total feature columns: {len(feature_cols)}')
print(f'Customers: {len(features)}')
print(f'  Train: {(features["split"]=="train").sum()}')
print(f'  Val:   {(features["split"]=="val").sum()}')
print(f'  Test:  {(features["split"]=="test").sum()}')
print(f'\nFeature list:')
for c in sorted(feature_cols):
    print(f'  {c}')

Saved features.parquet
Total feature columns: 58
Customers: 1200
  Train: 500
  Val:   200
  Test:  500

Feature list:
  avg_monthly_transaction_count
  avg_monthly_volume
  avg_txn_amount
  balance_to_inflow
  cash_total_volume
  cash_txn_count
  count_vs_baseline
  declined_ratio
  declined_txn_count
  dormancy_periods_count
  geographic_spread_score
  has_foreign_currency
  high_risk_country_txns
  income_declared_missing
  income_volume_ratio
  inflow_outflow_ratio
  kyc_medium
  max_counterparty_share
  max_single_transaction_6m
  max_single_txn
  max_vs_baseline
  min_cpi
  monthly_cnt_cv
  monthly_vol_cv
  nationality_cpi
  nationality_high_risk
  nationality_mismatch
  num_accounts_x
  num_accounts_y
  num_dormant
  num_unique_counterparties_6m
  pct_cash_deposits_of_cash
  pct_cash_transactions
  pct_cash_txns
  pct_cash_volume
  pct_dormant_accounts
  pct_international_transactions
  pct_intl_high_risk
  pct_intl_txns
  pct_offhours
  pct_recurring
  peak_7d_txn_count
  pep_s